# Optional VLM visual analysis and explanations

This notebook runs the implemented BLIP-based visual analysis and the separate consistency/explanation/integration contracts. It is intentionally optional: if VLM weights are unavailable, the notebook reports the limitation and still validates the structured interfaces. Captions are visual observations, not factual verification.

In [ ]:
%pip -q install -U torch transformers accelerate pandas pillow pyyaml matplotlib seaborn

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import torch

SMOKE_TEST = True
SAMPLE_COUNT = 4 if SMOKE_TEST else 32
VLM_MODEL_NAME = "Salesforce/blip-image-captioning-base"
REPO_ROOT_OVERRIDE = ""
candidates = [Path(REPO_ROOT_OVERRIDE)] if REPO_ROOT_OVERRIDE else []
candidates += [Path("/kaggle/working/FakeNewsDetector"), Path("/content/FakeNewsDetector"), Path.cwd()]
REPO_ROOT = next((p for p in candidates if (p / "src/models/proposed/vlm_analyzer.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT_OVERRIDE to the uploaded repository root.")
os.environ["PYTHONPATH"] = str(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
OUTPUT_ROOT = Path("/kaggle/working/research_outputs" if Path("/kaggle").exists() else "/content/research_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from PIL import Image
from src.models.proposed.vlm_analyzer import VLMAnalyzer
from src.models.proposed.consistency import ConsistencyLabel, assess_consistency
from src.models.proposed.integration import integrate_prediction

frame = pd.read_csv(REPO_ROOT / "data/splits/test.csv").head(SAMPLE_COUNT)
image_dir = REPO_ROOT / "data/raw/images"
try:
    analyzer = VLMAnalyzer(model_name=VLM_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
    outputs = []
    for _, row in frame.iterrows():
        image = Image.open(image_dir / str(row["image_url"])).convert("RGB")
        analysis = analyzer.analyze(str(row["title"]), image)
        consistency = assess_consistency(str(row["title"]), analysis.image_description, label=ConsistencyLabel.UNCERTAIN)
        integrated = integrate_prediction("UNKNOWN", None, vlm_analysis=analysis, consistency=consistency)
        outputs.append(integrated.to_dict())
    pd.DataFrame(outputs).to_json(OUTPUT_ROOT / "vlm_analysis.jsonl", orient="records", lines=True)
    display(pd.DataFrame([{"claim": x["vlm_analysis"]["claim"], "description": x["vlm_analysis"]["image_description"], "uncertainty": x["explanation"]["uncertainty_reasons"]} for x in outputs]))
except Exception as exc:
    print("VLM weights/backend unavailable; no visual results were generated:", type(exc).__name__, exc)
    print("Attach/download the BLIP checkpoint and rerun this cell.")

The notebook does not assign REAL/FAKE from the caption. To produce a complete integrated record, provide an independently trained classifier prediction and optional evidence source; otherwise the prediction remains UNKNOWN and the uncertainty is explicit.

# Visual outputs

The VLM notebook displays generated visual descriptions and plots the number of uncertainty reasons. This is an explanation/observation report, not a factuality score.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

vlm_path = OUTPUT_ROOT / "vlm_analysis.jsonl"
if vlm_path.exists():
    vlm_table = pd.read_json(vlm_path, lines=True)
    display(vlm_table[["claim", "description", "uncertainty"]] if "description" in vlm_table else vlm_table)
    if "uncertainty" in vlm_table:
        uncertainty_counts = vlm_table["uncertainty"].apply(lambda value: len(value) if isinstance(value, list) else 0).value_counts().sort_index()
        plt.figure(figsize=(7, 4))
        sns.barplot(x=uncertainty_counts.index.astype(str), y=uncertainty_counts.values, color="#e45756")
        plt.xlabel("Number of uncertainty reasons")
        plt.ylabel("Samples")
        plt.title("VLM uncertainty reasons per sample")
        plt.tight_layout()
        plt.savefig(OUTPUT_ROOT / "vlm_uncertainty_reasons.png", dpi=160)
        plt.show()
else:
    print("No VLM JSONL found; run the optional BLIP analysis cell first.")